# 🎯 Notebook 02 — IVP: YOLOv8 Object Detection Model
### Smart Retail Shelf — IVP + AI Team Project

**Owner role:** Vision (CNN) Lead

This notebook covers the full Computer Vision pipeline:

1. **Setup** — install `ultralytics`, load pretrained `yolov8n.pt`.
2. **Mock Dataset** — synthesize a tiny YOLO-format dataset (images +
   YOLO `.txt` labels) with a `data.yaml` config, so the notebook is fully
   self-contained and never depends on external downloads.
3. **Training** — fine-tune YOLOv8 on the mock dataset for 5 epochs.
4. **Evaluation & Inference** — run `model.predict()`, draw bounding boxes,
   save result images, and plot training metrics.

> ⚠️ **Colab tip:** Runtime ➜ Change runtime type ➜ **GPU (T4)** will make
> training dramatically faster. The notebook still runs correctly on CPU
> (just slower) since epochs=5 on a tiny mock dataset is intentionally
> lightweight.


## 0. Setup — Install Ultralytics & Imports

In [ ]:
# ============================================================
# 0.1  Install YOLOv8 (Ultralytics) — quiet install for Colab
# ============================================================
!pip install -q ultralytics opencv-python-headless matplotlib pyyaml


In [ ]:
# ============================================================
# 0.2  Imports
# ============================================================
import os
import shutil
import random
from pathlib import Path

import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Ultralytics + dependencies imported successfully ✅")


## 1. Pipeline Setup — Load Pretrained YOLOv8n

`yolov8n.pt` ("nano") is the smallest/fastest YOLOv8 checkpoint — ideal for
a course project and CPU-friendly fine-tuning demos. Ultralytics
auto-downloads the weights on first use.


In [ ]:
# ============================================================
# 1.1  Load the pretrained YOLOv8-nano checkpoint
#      (auto-downloads yolov8n.pt from Ultralytics on first run)
# ============================================================
model = YOLO("yolov8n.pt")

print("Loaded model:", model.model_name if hasattr(model, "model_name") else "yolov8n.pt")
print("Task:", model.task)


## 2. Mock "Smart Retail Shelf" Dataset

Real retail-shelf datasets (e.g., SKU110K) are large external downloads.
To keep this notebook **self-contained and error-free out-of-the-box**, we
programmatically generate a tiny synthetic dataset:

- Images: shelf-like backgrounds with colored rectangles = "products".
- Labels: YOLO-format `.txt` files (`class x_center y_center width height`,
  all normalized 0–1) generated from the exact rectangles we drew — so
  labels are always perfectly correct.
- 3 classes: `bottle`, `box`, `can`.

Folder layout follows the standard Ultralytics expectation:
```
retail_shelf_dataset/
├── images/{train,val}/*.jpg
├── labels/{train,val}/*.txt
└── data.yaml
```


In [ ]:
# ============================================================
# 2.1  Build a tiny synthetic YOLO-format detection dataset
# ============================================================
DATASET_DIR = Path("retail_shelf_dataset")
CLASS_NAMES = ["bottle", "box", "can"]
IMG_SIZE = 640

def make_dirs():
    for split in ["train", "val"]:
        (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def draw_shape(img, cls_id, x1, y1, x2, y2, rng):
    """Draw a class-specific shape so images look visually distinct."""
    color = [(66, 133, 244), (52, 168, 83), (234, 67, 53)][cls_id]
    if cls_id == 0:      # bottle -> tall thin rectangle
        cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
    elif cls_id == 1:    # box -> square-ish rectangle
        cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
    else:                # can -> filled ellipse
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        cv2.ellipse(img, (cx, cy), ((x2 - x1) // 2, (y2 - y1) // 2), 0, 0, 360, color, -1)
    cv2.rectangle(img, (x1, y1), (x2, y2), (20, 20, 20), 2)

def generate_sample(rng, n_objects=5):
    """Generate one synthetic shelf image + YOLO-format label lines."""
    img = np.full((IMG_SIZE, IMG_SIZE, 3), (240, 240, 240), dtype=np.uint8)
    cv2.rectangle(img, (0, IMG_SIZE - 40), (IMG_SIZE, IMG_SIZE), (60, 110, 160), -1)  # shelf strip

    labels = []
    x_cursor = 20
    for _ in range(n_objects):
        cls_id = int(rng.integers(0, len(CLASS_NAMES)))
        w = int(rng.integers(60, 110))
        h = int(rng.integers(80, 160))
        x1 = min(x_cursor, IMG_SIZE - w - 5)
        y2 = IMG_SIZE - 40
        y1 = max(y2 - h, 0)
        x2 = x1 + w
        draw_shape(img, cls_id, x1, y1, x2, y2, rng)

        # Convert absolute box -> normalized YOLO format (xc, yc, w, h)
        xc = (x1 + x2) / 2 / IMG_SIZE
        yc = (y1 + y2) / 2 / IMG_SIZE
        nw = (x2 - x1) / IMG_SIZE
        nh = (y2 - y1) / IMG_SIZE
        labels.append(f"{cls_id} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")

        x_cursor += w + int(rng.integers(15, 40))
        if x_cursor > IMG_SIZE - 60:
            break
    return img, labels

def build_dataset(n_train=16, n_val=4, seed=SEED):
    make_dirs()
    rng = np.random.default_rng(seed)
    for split, n in [("train", n_train), ("val", n_val)]:
        for i in range(n):
            img, labels = generate_sample(rng)
            img_path = DATASET_DIR / "images" / split / f"{split}_{i:03d}.jpg"
            lbl_path = DATASET_DIR / "labels" / split / f"{split}_{i:03d}.txt"
            cv2.imwrite(str(img_path), img)
            lbl_path.write_text("\n".join(labels))
    print(f"Dataset built -> {n_train} train / {n_val} val images at '{DATASET_DIR}/'")

build_dataset()


### 2.2 Create `data.yaml` Configuration

In [ ]:
# ============================================================
# 2.2  Write the YOLO data.yaml config that model.train() will read
# ============================================================
data_yaml = {
    "path": str(DATASET_DIR.resolve()),   # dataset root
    "train": "images/train",              # relative to 'path'
    "val": "images/val",
    "nc": len(CLASS_NAMES),                # number of classes
    "names": CLASS_NAMES,
}

yaml_path = DATASET_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"data.yaml written to: {yaml_path}\n")
print(yaml_path.read_text())


### 2.3 Sanity-check a Sample Image + Ground-Truth Boxes

In [ ]:
# ============================================================
# 2.3  Visualize one training sample with its label boxes overlaid
#      (quick sanity check that image <-> label generation lines up)
# ============================================================
sample_img_path = DATASET_DIR / "images" / "train" / "train_000.jpg"
sample_lbl_path = DATASET_DIR / "labels" / "train" / "train_000.txt"

sample_img = cv2.cvtColor(cv2.imread(str(sample_img_path)), cv2.COLOR_BGR2RGB)
h, w = sample_img.shape[:2]

vis = sample_img.copy()
for line in sample_lbl_path.read_text().strip().split("\n"):
    cls_id, xc, yc, bw, bh = map(float, line.split())
    x1 = int((xc - bw / 2) * w); y1 = int((yc - bh / 2) * h)
    x2 = int((xc + bw / 2) * w); y2 = int((yc + bh / 2) * h)
    cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
    cv2.putText(vis, CLASS_NAMES[int(cls_id)], (x1, max(y1 - 6, 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

plt.figure(figsize=(5, 5))
plt.imshow(vis)
plt.title("Sample training image with ground-truth boxes")
plt.axis("off")
plt.show()


## 3. Model Training

Fine-tunes `yolov8n.pt` on the mock dataset for **5 epochs**. With such a
tiny synthetic dataset this is purely a pipeline demonstration (not a
production-quality model) — swap `data.yaml` to point at a real annotated
shelf dataset for actual deployment.


In [ ]:
# ============================================================
# 3.1  Train YOLOv8 for 5 epochs on the mock retail-shelf dataset
# ============================================================
train_results = model.train(
    data=str(yaml_path),
    epochs=5,
    imgsz=IMG_SIZE,
    batch=4,
    project="runs_retail_shelf",
    name="yolov8n_retail",
    exist_ok=True,
    patience=0,      # no early stopping needed for a 5-epoch demo
    verbose=True,
)

print("Training complete. Results saved to:", train_results.save_dir)


## 4. Model Evaluation & Inference

### 4.1 Run Inference on Test Images and Draw Bounding Boxes


In [ ]:
# ============================================================
# 4.1  Load the best fine-tuned checkpoint & run inference
# ============================================================
best_weights = Path(train_results.save_dir) / "weights" / "best.pt"
trained_model = YOLO(str(best_weights)) if best_weights.exists() else model

# Run predictions on the validation images as our "test set"
test_images_dir = DATASET_DIR / "images" / "val"
predict_results = trained_model.predict(
    source=str(test_images_dir),
    imgsz=IMG_SIZE,
    conf=0.15,          # low threshold since this is a tiny demo model
    save=True,           # saves annotated images to disk
    project="runs_retail_shelf",
    name="predictions",
    exist_ok=True,
)

print(f"Ran inference on {len(predict_results)} image(s).")
print(f"Annotated result images saved under: runs_retail_shelf/predictions/")


### 4.2 Display Detected Bounding Boxes Inline

In [ ]:
# ============================================================
# 4.2  Plot detections for each test image (boxes drawn by Ultralytics)
# ============================================================
n_show = min(4, len(predict_results))
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
if n_show == 1:
    axes = [axes]

for ax, result in zip(axes, predict_results[:n_show]):
    annotated_bgr = result.plot()  # numpy array (BGR) with boxes/labels drawn
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    n_dets = len(result.boxes) if result.boxes is not None else 0
    ax.set_title(f"{Path(result.path).name}\n{n_dets} detection(s)")
    ax.axis("off")

plt.suptitle("YOLOv8 Inference — Detected Items on Shelf", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


### 4.3 Save Annotated Result Images Explicitly

In [ ]:
# ============================================================
# 4.3  Explicitly (re)save each annotated result image to a clean folder
#      (Ultralytics already saved them via save=True above; this shows
#      how to do it manually too, e.g. for custom naming/postprocessing)
# ============================================================
OUTPUT_DIR = Path("saved_predictions")
OUTPUT_DIR.mkdir(exist_ok=True)

for result in predict_results:
    annotated_bgr = result.plot()
    out_name = OUTPUT_DIR / f"annotated_{Path(result.path).name}"
    cv2.imwrite(str(out_name), annotated_bgr)

print(f"Saved {len(predict_results)} annotated image(s) to '{OUTPUT_DIR}/':")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)


### 4.4 Plot Training Metrics (Loss Curves / mAP)

In [ ]:
# ============================================================
# 4.4  Plot metrics logged during training
#      Ultralytics writes results.csv into the run's save_dir
# ============================================================
import pandas as pd

results_csv = Path(train_results.save_dir) / "results.csv"

if results_csv.exists():
    metrics_df = pd.read_csv(results_csv)
    metrics_df.columns = [c.strip() for c in metrics_df.columns]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Loss curves ---
    loss_cols = [c for c in metrics_df.columns if "loss" in c.lower()]
    for col in loss_cols:
        axes[0].plot(metrics_df["epoch"], metrics_df[col], marker="o", label=col)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss Curves"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    # --- mAP curves (if present) ---
    map_cols = [c for c in metrics_df.columns if "map" in c.lower()]
    for col in map_cols:
        axes[1].plot(metrics_df["epoch"], metrics_df[col], marker="s", label=col)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mAP")
    axes[1].set_title("Validation mAP"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    plt.suptitle("YOLOv8 Training Metrics — 5 Epochs", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print("\nFinal-epoch metrics:")
    print(metrics_df.tail(1).T)
else:
    print("results.csv not found — metrics plotting skipped (training may not have produced logs).")


---
## ✅ Notebook 02 Summary

| Step | Artifact |
|---|---|
| Pretrained load | `yolov8n.pt` |
| Mock dataset | `retail_shelf_dataset/` (images + YOLO labels + `data.yaml`) |
| Fine-tuned weights | `runs_retail_shelf/yolov8n_retail/weights/best.pt` |
| Predictions | `runs_retail_shelf/predictions/` and `saved_predictions/` |
| Metrics | `results.csv` + loss/mAP plots |

**Next:** the AI Lead consumes co-purchase patterns in
`ai_correlation_knn.ipynb`; detected item classes from this notebook could
be joined with basket data in a later integration milestone.
